# Build a PDF Question-Answering RAG Chain

**Course author:** Jayneel Girnara

Build a retrieval-augmented generation (RAG) pipeline that loads a PDF, indexes its chunks, retrieves relevant context, and asks a chat model to answer from that context.

## Learning goals
- Load and split a PDF into overlapping chunks.
- Embed and index the chunks in Chroma.
- Retrieve relevant passages for a question.
- Compose retrieval context with a prompt and chat model.

## Before you start
Install dependencies, place `data_science_syllabus.pdf` in `data/`, and configure `OPENAI_API_KEY` in .env. Embedding and chat calls may incur charges. Keep API keys private.

## How to work through it
Run each stage top to bottom and inspect the retrieved context before calling the model. Ask a question answerable from the document, then compare the final response against the retrieved passages. Treat the output as grounded only when the evidence supports it.

## 1. Load and split the PDF

First, load each PDF page as a `Document`, then split its text into overlapping chunks. These chunks are the units that will be embedded and searched. Page/source metadata lets you trace an answer back to the PDF.

**Checkpoint:** Why do we split before embedding and retrieval?

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent

load_dotenv(project_root / ".env")


In [21]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate


In [ ]:
loader=PyPDFLoader(project_root / "data" / "data_science_syllabus.pdf")
docs=loader.load()
len(docs)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data=splitter.split_documents(documents=docs)
len(splitted_data)

## 2. Embed chunks and create the vector store

The embedding model converts every chunk into a numeric vector. Chroma stores those vectors with the corresponding text so they can be searched later. This embedding call uses OpenAI and may incur charges.

**Checkpoint:** What does the vector store need to keep besides the numeric vector?

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")


In [ ]:
vector_store=Chroma.from_documents(documents=splitted_data,
                                   embedding=embeddings,
                                   )

## 3. Retrieve relevant context

A **retriever** uses a question to find likely relevant chunks. Inspect these results: if they do not contain the evidence, the model cannot reliably answer from the PDF. More results are not always better; irrelevant passages can distract the model.

**Checkpoint:** What would you do if the retrieved chunks do not contain the answer?

In [ ]:
query="Machine learning and data science content"
data=vector_store.similarity_search(query=query)


In [ ]:
context=""
for doc in data:
    context+=doc.page_content+"\n"

In [ ]:
llm=ChatOpenAI(model="gpt-4o-mini")

## RAG response generation

Use the retrieved passages as context for a prompt, then pass the prompt through the chat model.

Grounding means providing source passages to the model as context; it does not guarantee the model will follow them. A clear prompt should instruct it to acknowledge when the passages do not answer the question.

**Checkpoint:** What should the model say if the supplied context is insufficient?

## 4. Build and invoke the RAG chain

This chain connects the retrieval function, prompt template, and chat model. Its input is the question; the function builds a context/question mapping; the prompt formats it; then the model generates a response. Compare that response with the actual retrieved passages.

**Checkpoint:** Which chain step retrieves the source text, and which step generates the answer?

In [22]:
def get_context(query:str):
    data=vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context+=doc.page_content+"\n"
    return {
        "context":context,
        "question":query
    }


    

In [23]:
prompt=PromptTemplate.from_template("""
       You are a helpful assistant and provide answer based on the context for user question. and 
       if you don't know the answer,then you can say I dont know the answer
       context:{context}
       Queston:{question}
""")

In [24]:
rag_chain=get_context|prompt|llm

In [27]:
res=rag_chain.invoke("Give me complete content for gen ai module")
print(res.content)

Here is the complete GenAI module content from your context, consolidated and organized:

Module 10: GenAI Foundations (4 weeks, Month 10)
- Topics:
  - What is Generative AI and Large Language Models (LLMs)
  - Prompt engineering
  - Embeddings and Vector Databases (FAISS)
  - OpenAI API, Anthropic, Gemini
- Tools:
  - OpenAI, ChatGPT, Claude, Gemini, Replit, Google Colab
- Mini Projects:
  - Build a chatbot using OpenAI
  - Auto-summarizer using ChatGPT API
- Mock Interview:
  - Prompting, GenAI APIs, RAG Concept

Module 11: GenAI App Building (4 weeks, Month 11)
- Topics:
  - LangChain basics
  - LlamaIndex and RAG pipeline
  - Build workflows with Flowise
  - Deploy with Streamlit/Gradio
- Tools:
  - LangChain, Streamlit, Hugging Face, Vercel
- Mini Projects:
  - Resume Q&A bot using LangChain
  - Voice-to-Summary App using Whisper + GPT
- Mock Interview:
  - LLM App Architecture + RAG Debugging

Module 12: GenAI Tools Lab & AI Playground (Month 11)
- Goal:
  - Hands-on practice wi